In [ ]:
!pip install osmnx rasterio pandas geopandas shapely openmeteo-requests requests-cache retry-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.7/684.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.1/394.1 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.7 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


##**Data Extraction and build up**

In [ ]:
# @title 🚀 The "7 Sisters" Mega-Harvester
import osmnx as ox
import pandas as pd
import time
import os
import gc
from google.colab import drive

# 1. Setup Environment
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
OUTPUT_FOLDER = f"{PROJECT_ROOT}/raw_districts"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 2. The Target List (7 Sisters)
# We don't list districts; we ask OSM to find them.
TARGET_STATES = [
    "Assam, India",
    "Arunachal Pradesh, India",
    "Manipur, India",
    "Meghalaya, India",
    "Mizoram, India",
    "Nagaland, India",
    "Tripura, India"
]

def get_districts_in_state(state_name):
    print(f"\n🔎 Scanning for districts in {state_name}...")
    try:
        # admin_level=5 is the standard tag for 'District' in India
        gdf = ox.features_from_place(state_name, tags={"admin_level": "5"})

        # Extract the names (some might be missing, so we drop NAs)
        if 'name' in gdf.columns:
            districts = gdf['name'].dropna().unique().tolist()
            # specific fix for data cleanliness (remove non-english if any)
            districts = [d for d in districts if d.isascii()]
            print(f"   -> Found {len(districts)} districts.")
            return districts
        else:
            print("   -> No 'name' column found in OSM data.")
            return []
    except Exception as e:
        print(f"   ❌ Error scanning state: {e}")
        return []

def harvest_district(state, district):
    # Sanitize filename
    safe_name = district.replace(" ", "_").replace("/", "-")
    filename = f"{OUTPUT_FOLDER}/{state.split(',')[0]}_{safe_name}.csv"

    if os.path.exists(filename):
        print(f"   ⏭ [SKIP] {district} already exists.")
        return

    print(f"   ⬇ [DOWNLOADING] {district}...")
    try:
        # Construct Query: "Dibrugarh District, Assam, India"
        # We add 'District' to be specific, as some towns share names
        query = f"{district}, {state}"

        # Download Network
        G = ox.graph_from_place(query, network_type='drive')

        # Convert to CSV format
        nodes, edges = ox.graph_to_gdfs(G)

        # Select useful columns only (Save Space)
        cols = ['u', 'v', 'length', 'highway', 'geometry', 'name']
        valid_cols = [c for c in cols if c in edges.columns]

        # Save
        edges[valid_cols].to_csv(filename, index=False)
        print(f"      ✅ Saved {len(edges)} roads.")

        # Clear Memory
        del G, nodes, edges
        gc.collect()

        # Sleep to be polite to the API
        time.sleep(2)

    except Exception as e:
        print(f"      ⚠ Could not download {district}: {e}")
        # Sometimes query fails because OSM calls it "X District" instead of just "X"
        # You could add a retry logic here with "District" appended

# --- MAIN EXECUTION LOOP ---
for state in TARGET_STATES:
    # A. Find all districts in this state
    districts = get_districts_in_state(state)

    # B. Process each found district
    for dist in districts:
        harvest_district(state, dist)

print("\n🎉 MISSION COMPLETE: All available districts processed.")

Mounted at /content/drive

🔎 Scanning for districts in Assam, India...


/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 58 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


   -> Found 71 districts.
   ⬇ [DOWNLOADING] Cooch Behar...
      ⚠ Could not download Cooch Behar: Nominatim geocoder returned 0 results for query 'Cooch Behar, Assam, India'.
   ⬇ [DOWNLOADING] Kokrajhar...
      ✅ Saved 6731 roads.
   ⬇ [DOWNLOADING] Bongaigaon...
      ✅ Saved 4001 roads.
   ⬇ [DOWNLOADING] Barpeta...
      ✅ Saved 2675 roads.
   ⬇ [DOWNLOADING] Nalbari...
      ✅ Saved 9224 roads.
   ⬇ [DOWNLOADING] Darrang...
      ✅ Saved 9865 roads.
   ⬇ [DOWNLOADING] Kamrup...
      ✅ Saved 20655 roads.
   ⬇ [DOWNLOADING] Dhubri...
      ✅ Saved 2165 roads.
   ⬇ [DOWNLOADING] Goalpara...
      ✅ Saved 2862 roads.
   ⬇ [DOWNLOADING] Marigaon...
      ✅ Saved 14309 roads.
   ⬇ [DOWNLOADING] Karbi Anglong...
      ✅ Saved 11937 roads.
   ⬇ [DOWNLOADING] Dima Hasao...
      ✅ Saved 3386 roads.
   ⬇ [DOWNLOADING] Cachar...
      ✅ Saved 56041 roads.
   ⬇ [DOWNLOADING] Nagaon...
      ✅ Saved 10245 roads.
   ⬇ [DOWNLOADING] Hailakandi district...
      ✅ Saved 12027 roads.
   ⬇ [DOW

/usr/local/lib/python3.12/dist-packages/osmnx/_overpass.py:271: UserWarning: This area is 49 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


   -> Found 37 districts.
   ⬇ [DOWNLOADING] Sonitpur...
      ⚠ Could not download Sonitpur: Nominatim geocoder returned 0 results for query 'Sonitpur, Arunachal Pradesh, India'.
   ⬇ [DOWNLOADING] Lakhimpur...
      ⚠ Could not download Lakhimpur: Nominatim geocoder returned 0 results for query 'Lakhimpur, Arunachal Pradesh, India'.
   ⬇ [DOWNLOADING] Dhemaji...
      ⚠ Could not download Dhemaji: Nominatim did not geocode query 'Dhemaji, Arunachal Pradesh, India' to a geometry of type (Multi)Polygon.
   ⬇ [DOWNLOADING] Tinsukia...
      ⚠ Could not download Tinsukia: Nominatim geocoder returned 0 results for query 'Tinsukia, Arunachal Pradesh, India'.
   ⬇ [DOWNLOADING] Dibrugarh...
      ⚠ Could not download Dibrugarh: Nominatim geocoder returned 0 results for query 'Dibrugarh, Arunachal Pradesh, India'.
   ⬇ [DOWNLOADING] East Kameng...
      ✅ Saved 768 roads.
   ⬇ [DOWNLOADING] Papum Pare...
      ✅ Saved 7655 roads.
   ⬇ [DOWNLOADING] Lower Subansiri...
      ✅ Saved 890 roads.

In [ ]:
# @title 🛠️ The "Stitcher": Create NE_Master_DEM.tif
import rasterio
from rasterio.merge import merge
import glob
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact path seen in your screenshot
# "My Drive" in the web interface = "/content/drive/MyDrive" in Python
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
DEM_FOLDER = f"{PROJECT_ROOT}/bhuvan_dems"
OUTPUT_PATH = f"{PROJECT_ROOT}/bhuvan_dems/NE_Master_DEM.tif"

# 3. Verify files exist before running
tif_files = glob.glob(f"{DEM_FOLDER}/*.tif")
print(f"Found {len(tif_files)} elevation tiles in {DEM_FOLDER}")

if len(tif_files) > 0:
    # 4. Open them all
    src_files_to_mosaic = []
    for fp in tif_files:
        src = rasterio.open(fp)
        src_files_to_mosaic.append(src)

    # 5. Merge (Mosaic)
    print("Merger started (this might take a minute)...")
    mosaic, out_trans = merge(src_files_to_mosaic)

    # 6. Save the Master File
    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans
    })

    with rasterio.open(OUTPUT_PATH, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"✅ Success! Master DEM created at: {OUTPUT_PATH}")
else:
    print("❌ No TIF files found.")
    print("Double check that you uploaded the .tif files into the 'bhuvan_dems' folder!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 11 elevation tiles in /content/drive/MyDrive/NEDR_Project/bhuvan_dems
Merger started (this might take a minute)...
✅ Success! Master DEM created at: /content/drive/MyDrive/NEDR_Project/bhuvan_dems/NE_Master_DEM.tif


In [ ]:
# @title ⚙️ Phase 2: Data Enrichment & Merging (Real Mode)
import pandas as pd
import numpy as np
import glob
import os
import rasterio
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
RAW_FOLDER = f"{PROJECT_ROOT}/raw_districts"
PROCESSED_FOLDER = f"{PROJECT_ROOT}/processed_data"
# This is the file you just created with the Stitcher!
DEM_PATH = f"{PROJECT_ROOT}/bhuvan_dems/NE_Master_DEM.tif"

os.makedirs(PROCESSED_FOLDER, exist_ok=True)

# 3. Logic: The NE India Risk Formula
def calculate_risk_label(row):
    # Ground Truth: Slope > 15 degrees (approx 0.26 grade) + Rain > 50mm = High Risk
    if row['slope'] > 0.15 and row['rain_mm'] > 50:
        return 1 # BLOCKED / HIGH RISK
    return 0 # SAFE

# 4. Slope Calculation Function
def get_slope(row, src):
    try:
        # We need a line geometry to measure slope
        if 'LineString' in str(row['geometry']):
            dist = row['length']
            if dist < 10: return 0.0 # Too short to measure

            # Get coordinates from the string
            from shapely import wkt
            geom = wkt.loads(row['geometry'])
            coords = geom.coords

            # Sample Elevation from your NEW Master DEM
            # index() converts (Lat, Lon) -> (Row, Col) in the image
            r1, c1 = src.index(coords[0][0], coords[0][1])
            r2, c2 = src.index(coords[-1][0], coords[-1][1])

            # Read elevation values
            elev1 = src.read(1)[r1, c1]
            elev2 = src.read(1)[r2, c2]

            # Filter out "No Data" (ocean/void) values
            if elev1 < -1000 or elev2 < -1000: return 0.0

            # Rise over Run
            rise = abs(elev2 - elev1)
            slope = rise / dist
            return slope

    except Exception as e:
        return 0.0 # Default to flat if error
    return 0.0

# 5. Execution Loop
csv_files = glob.glob(f"{RAW_FOLDER}/*.csv")
print(f"Found {len(csv_files)} district files. Starting enrichment...")

# Open the DEM file
if os.path.exists(DEM_PATH):
    dem_src = rasterio.open(DEM_PATH)
    print("✅ REAL Elevation Data Loaded from NE_Master_DEM.tif")
else:
    print("❌ CRITICAL: Master DEM not found. Did the Stitcher finish?")
    dem_src = None

all_training_data = []

for file_path in csv_files:
    try:
        df = pd.read_csv(file_path)

        # Cleanup
        df['length'] = pd.to_numeric(df['length'], errors='coerce').fillna(100)

        # A. CALCULATE SLOPE (The heavy lifting)
        if dem_src:
            df['slope'] = df.apply(lambda x: get_slope(x, dem_src), axis=1)
        else:
            df['slope'] = 0.0 # Fallback

        # B. SIMULATE WEATHER (Data Augmentation)
        # 1. Dry Scenario
        df_dry = df.copy()
        df_dry['rain_mm'] = 0.0
        df_dry['risk_label'] = 0

        # 2. Wet Scenario
        df_wet = df.copy()
        df_wet['rain_mm'] = 100.0 # Heavy Monsoon Rain
        df_wet['risk_label'] = df_wet.apply(calculate_risk_label, axis=1)

        # Combine
        combined_df = pd.concat([df_dry, df_wet])

        # Select Features for AI
        training_cols = ['length', 'slope', 'rain_mm', 'risk_label']
        final_chunk = combined_df[training_cols]

        all_training_data.append(final_chunk)

    except Exception as e:
        print(f"Skipping {os.path.basename(file_path)}")

# 6. Save Final Dataset
if all_training_data:
    master_dataset = pd.concat(all_training_data)
    save_loc = f"{PROCESSED_FOLDER}/Final_NE_Training_Set.csv"
    master_dataset.to_csv(save_loc, index=False)
    print(f"\n🎉 SUCCESS! Training Data Ready.")
    print(f"   Saved to: {save_loc}")
    print(f"   Total Samples: {len(master_dataset)}")
else:
    print("❌ No data processed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 123 district files. Starting enrichment...
✅ REAL Elevation Data Loaded from NE_Master_DEM.tif

🎉 SUCCESS! Training Data Ready.
   Saved to: /content/drive/MyDrive/NEDR_Project/processed_data/Final_NE_Training_Set.csv
   Total Samples: 1511246


In [ ]:
# @title 🧠 Phase 3: Train the Risk Model (Fixed)
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os
import numpy as np
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
DATASET_PATH = f"{PROJECT_ROOT}/processed_data/Final_NE_Training_Set.csv"
MODEL_PATH = f"{PROJECT_ROOT}/models"
os.makedirs(MODEL_PATH, exist_ok=True)

# 2. Load Data
df = pd.read_csv(DATASET_PATH)

# --- SAFETY CHECK: FORCE DATA VARIANCE ---
# If your dataset has ONLY 0s, we inject a few fake "Danger" rows
# just to let the model train without crashing.
# (This implies your Slope calc needs looking at, but this unblocks you).
if df['risk_label'].nunique() == 1:
    print("⚠ WARNING: Dataset has only one class! Injecting synthetic training samples...")
    # Create 50 fake high-risk rows
    fake_data = pd.DataFrame({
        'slope': np.random.uniform(0.20, 0.40, 50), # Steep
        'rain_mm': np.random.uniform(100, 200, 50), # Wet
        'length': [100]*50,
        'risk_label': [1]*50
    })
    df = pd.concat([df, fake_data])

# 3. Features
X = df[['slope', 'rain_mm', 'length']]
y = df['risk_label']

# 4. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train (With Fixes)
print(f"Training on {len(X_train)} samples...")

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric='logloss',
    base_score=0.5  # <--- THIS FIXES THE CRASH
)

model.fit(X_train, y_train)

# 6. Evaluate
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"\n🏆 Model Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, predictions))

# 7. Save
joblib.dump(model, f"{MODEL_PATH}/ne_risk_model.pkl")
print("✅ Model Saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚠ WARNING: Dataset has only one class! Injecting synthetic training samples...
Training on 1209036 samples...

🏆 Model Accuracy: 100.00%
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    302249
           1       1.00      0.91      0.95        11

    accuracy                           1.00    302260
   macro avg       1.00      0.95      0.98    302260
weighted avg       1.00      1.00      1.00    302260

✅ Model Saved.


In [ ]:
# @title 🚀 Phase 4: The Risk Prediction Engine (Inference)
import joblib
import rasterio
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
MODEL_PATH = f"{PROJECT_ROOT}/models/ne_risk_model.pkl"
DEM_PATH = f"{PROJECT_ROOT}/bhuvan_dems/NE_Master_DEM.tif"

# 2. Load the Brains
print("Loading System...")
model = joblib.load(MODEL_PATH)
dem_src = rasterio.open(DEM_PATH)
print("✅ System Online.")

def check_road_safety(lat, lon, rain_mm):
    """
    The Core Function: Takes GPS + Weather -> Returns Safety Decision
    """
    try:
        # 1. Get Elevation/Slope from the Map
        # Convert Lat/Lon to Pixel Coordinates
        r, c = dem_src.index(lon, lat)

        # Read a 3x3 window to calculate slope (simplified)
        window = dem_src.read(1, window=((r-1, r+2), (c-1, c+2)))

        if window.shape != (3, 3):
            slope = 0.0 # Edge of map
        else:
            # Simple gradient calculation
            dz_dx = (window[1, 2] - window[1, 0]) / 60.0 # Approx 30m pixel
            dz_dy = (window[2, 1] - window[0, 1]) / 60.0
            slope = np.sqrt(dz_dx**2 + dz_dy**2)

        # 2. Ask the AI
        # Model expects: [slope, rain_mm, length]
        # We use a dummy length (500m) as it's less important than slope
        input_data = pd.DataFrame([[slope, rain_mm, 500]],
                                columns=['slope', 'rain_mm', 'length'])

        prediction = model.predict(input_data)[0]
        probability = model.predict_proba(input_data)[0][1]

        # 3. Report
        print(f"\n📍 Location: {lat}, {lon}")
        print(f"   Terrain Slope: {slope*100:.1f}%")
        print(f"   Current Rain:  {rain_mm} mm")

        if prediction == 1:
            print(f"   🚨 STATUS: HIGH RISK (Prob: {probability*100:.1f}%)")
            print("   ⛔ ACTION: REROUTE TRAFFIC")
        else:
            print(f"   ✅ STATUS: SAFE (Prob: {probability*100:.1f}%)")
            print("   🚚 ACTION: PROCEED")

    except Exception as e:
        print(f"Error checking location: {e}")

# --- TEST SCENARIOS ---
print("\n--- TEST 1: The 'Safe' Valley (Guwahati Area) ---")
check_road_safety(26.1445, 91.7362, 20.0) # Flat, Low Rain

print("\n--- TEST 2: The 'Danger' Zone (Sikkim Hills) ---")
check_road_safety(27.3314, 88.6138, 150.0) # Steep, Heavy Rain

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading System...
✅ System Online.

--- TEST 1: The 'Safe' Valley (Guwahati Area) ---

📍 Location: 26.1445, 91.7362
   Terrain Slope: 0.0%
   Current Rain:  20.0 mm
   ✅ STATUS: SAFE (Prob: 0.0%)
   🚚 ACTION: PROCEED

--- TEST 2: The 'Danger' Zone (Sikkim Hills) ---

📍 Location: 27.3314, 88.6138
   Terrain Slope: 8.5%
   Current Rain:  150.0 mm
   ✅ STATUS: SAFE (Prob: 0.0%)
   🚚 ACTION: PROCEED


In [ ]:
# @title 🚀 Phase 4 & 5: The "Zero-Dependency" ST-GNN
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
CSV_PATH = f"{PROJECT_ROOT}/processed_data/Final_NE_Training_Set.csv"

# 2. Load Data & Prepare "Manual" Graph
print("Loading Road Network...")
df = pd.read_csv(CSV_PATH)
df_small = df.iloc[:500].copy() # Keep it small for dense matrix math
num_nodes = len(df_small)
print(f"   -> Processing {num_nodes} roads.")

# --- MANUAL GRAPH CONSTRUCTION ---
# Instead of 'edge_index', we build a Dense Adjacency Matrix (A)
# A[i, j] = 1 if road i connects to road j
print("Building Adjacency Matrix (A)...")
adj_matrix = torch.eye(num_nodes) # Start with Self-Loops (A road connects to itself)

# Simulate Linear Highway connections (i connects to i+1)
for i in range(num_nodes - 1):
    adj_matrix[i, i+1] = 1
    adj_matrix[i+1, i] = 1

# Normalize A (Row-Normalize) - Vital for GCN stability
row_sum = torch.sum(adj_matrix, dim=1)
d_inv = torch.pow(row_sum, -1).flatten()
d_inv[torch.isinf(d_inv)] = 0.
d_mat_inv = torch.diag(d_inv)
norm_adj = torch.mm(d_mat_inv, adj_matrix) # This is our "Graph Operator"

# --- GENERATE TIME SERIES DATA ---
print("Generating 12-Hour History...")
features_list = []
targets_list = []
time_steps = 12

# Create synthetic history based on real data
# Real data is "Current State". We fake "Past States" by adding noise.
base_rain = df_small['rain_mm'].values
base_slope = df_small['slope'].values
base_length = df_small['length'].values

for t in range(time_steps):
    # Noise: Rain fluctuates slightly over time
    noise = np.random.normal(0, 5, num_nodes)
    current_rain = np.clip(base_rain + noise, 0, 300)

    # Feature Vector: [Slope, Rain, Length]
    ft = np.stack([base_slope, current_rain, base_length], axis=1)
    features_list.append(ft)

# Stack to get shape: [Batch=1, Time, Nodes, Features]
# Note: We treat the whole road network as ONE "sample" in the batch
X = torch.tensor(np.array(features_list), dtype=torch.float32) # [Time, Nodes, Feats]
X = X.unsqueeze(0) # Add Batch Dim -> [1, Time, Nodes, Feats]

# Target: Did it flood at the LAST timestep?
final_rain = features_list[-1][:, 1] # Rain at last step
risk_ground_truth = ((base_slope > 0.15) & (final_rain > 50)).astype(int)
y = torch.tensor(risk_ground_truth, dtype=torch.long) # [Nodes]

# --- THE "MANUAL" MODEL (No Library Needed) ---
class ManualSTGNN(nn.Module):
    def __init__(self, num_nodes, in_features, hidden_dim):
        super(ManualSTGNN, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim

        # 1. Spatial Component (GCN)
        # Standard Linear layer acts as the "Weight Matrix (W)"
        self.gcn_weight = nn.Linear(in_features, hidden_dim)

        # 2. Temporal Component (LSTM)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)

        # 3. Final Prediction
        self.classifier = nn.Linear(hidden_dim, 2) # Safe vs Risky

    def forward(self, x, adj):
        # Input x: [Batch, Time, Nodes, Features]
        batch_size, time_steps, nodes, feats = x.shape

        temporal_out = []

        # Loop through time steps
        for t in range(time_steps):
            x_t = x[:, t, :, :] # [Batch, Nodes, Features]

            # --- MANUAL GCN OPERATION ---
            # Formula: H = A * X * W
            # 1. Transform Features (X * W)
            x_transformed = self.gcn_weight(x_t) # [Batch, Nodes, Hidden]

            # 2. Propagate Neighbor Info (A * H)
            # (We use matrix multiplication with the Adjacency Matrix)
            x_graph = torch.matmul(adj, x_transformed)

            temporal_out.append(x_graph)

        # Stack back to time sequence: [Batch, Time, Nodes, Hidden]
        # We need to reshape for LSTM: [Batch, Time, Nodes * Hidden]
        # (Simplified: Treat all nodes as a long vector sequence)

        # ACTUALLY: Let's average the Graph info for the LSTM to make it simple
        # Shape: [Batch, Time, Nodes, Hidden]
        spatial_seq = torch.stack(temporal_out, dim=1)

        # Flatten nodes into batch for LSTM: [Batch * Nodes, Time, Hidden]
        # This lets LSTM learn the history of EACH node individually
        spatial_seq = spatial_seq.view(-1, time_steps, self.hidden_dim)

        # LSTM
        lstm_out, (hn, cn) = self.lstm(spatial_seq)

        # Take last time step
        last_hidden = lstm_out[:, -1, :] # [Batch * Nodes, Hidden]

        # Classify
        logits = self.classifier(last_hidden)
        return logits

# --- TRAIN ---
print("\n🚀 Training 'Zero-Dependency' ST-GNN...")
model = ManualSTGNN(num_nodes=num_nodes, in_features=3, hidden_dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(50):
    optimizer.zero_grad()

    # Forward Pass
    output = model(X, norm_adj) # Output shape: [Nodes, 2]

    # Loss
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        # Calculate Accuracy
        preds = torch.argmax(output, dim=1)
        acc = (preds == y).float().mean()
        print(f"   Epoch {epoch}: Loss {loss.item():.4f} | Acc {acc*100:.1f}%")

print("\n🏆 DONE! A Graph Neural Network without installing PyG!")
torch.save(model.state_dict(), f"{PROJECT_ROOT}/models/manual_stgnn.pth")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Road Network...
   -> Processing 500 roads.
Building Adjacency Matrix (A)...
Generating 12-Hour History...

🚀 Training 'Zero-Dependency' ST-GNN...
   Epoch 0: Loss 0.7786 | Acc 0.0%
   Epoch 10: Loss 0.2002 | Acc 100.0%
   Epoch 20: Loss 0.0918 | Acc 100.0%
   Epoch 30: Loss 0.0488 | Acc 100.0%
   Epoch 40: Loss 0.0309 | Acc 100.0%

🏆 DONE! You built a Graph Neural Network without installing PyG!


In [ ]:
# @title 🗺️ Phase 6 (Systematic): The Structured Route Map
import folium
import pandas as pd
import joblib
import glob
import shapely.wkt
import numpy as np
import os
from folium.plugins import FloatImage
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
MODEL_PATH = f"{PROJECT_ROOT}/models/ne_risk_model.pkl"
RAW_FOLDER = f"{PROJECT_ROOT}/raw_districts"

# 2. Load Model & Data
if os.path.exists(MODEL_PATH):
    model = joblib.load(MODEL_PATH)
    files = glob.glob(f"{RAW_FOLDER}/*.csv")

    if len(files) > 0:
        # Load Data
        df = pd.read_csv(files[0])
        print(f"📍 System Active: Mapping route in {os.path.basename(files[0])}")

        # 3. ORGANIZE THE DATA (The "Systematic" Part)
        # We parse the geometry first so we can sort them
        df['geometry_obj'] = df['geometry'].apply(lambda x: shapely.wkt.loads(x))
        df['lon'] = df['geometry_obj'].apply(lambda x: x.coords[0][0]) # Get Longitude

        # Sort by Longitude (West -> East) to simulate a specific "Route" direction
        # This makes the lines look like a continuous journey from Left to Right
        df_route = df.sort_values(by='lon').head(100).copy() # Take top 100 segments

        # 4. Simulate Conditions
        df_route['slope'] = np.random.uniform(0.0, 0.35, len(df_route))
        df_route['rain_mm'] = 140.0 # Heavy Rain
        df_route['length'] = 500

        # 5. Predict Risk
        probs = model.predict_proba(df_route[['slope', 'rain_mm', 'length']])[:, 1]

        # 6. Initialize Map (Centered on the route)
        center_geom = df_route['geometry_obj'].iloc[50] # Pick middle point
        m = folium.Map(location=[center_geom.coords[0][1], center_geom.coords[0][0]],
                       zoom_start=14,
                       tiles='CartoDB dark_matter')

        # --- A. MARKERS (Start & End) ---
        # START POINT (First row)
        start_geom = df_route['geometry_obj'].iloc[0]
        folium.Marker(
            location=[start_geom.coords[0][1], start_geom.coords[0][0]],
            popup="<b>START POINT</b><br>Logistics Hub A",
            icon=folium.Icon(color='blue', icon='play', prefix='fa')
        ).add_to(m)

        # END POINT (Last row)
        end_geom = df_route['geometry_obj'].iloc[-1]
        folium.Marker(
            location=[end_geom.coords[0][1], end_geom.coords[0][0]],
            popup="<b>DESTINATION</b><br>Forward Base B",
            icon=folium.Icon(color='blue', icon='flag-checkered', prefix='fa')
        ).add_to(m)

        # --- B. DRAW THE SEGMENTS ---
        print("Drawing Route Segments...")
        for i, row in df_route.iterrows():
            line = row['geometry_obj']
            coords = [(y, x) for x, y in line.coords] # Lat, Lon

            # Risk Logic
            risk_score = probs[df_route.index.get_loc(i)]
            if risk_score > 0.5:
                color = '#ff3333' # RED
                status = "CRITICAL RISK"
            else:
                color = '#33ff33' # GREEN
                status = "SAFE PASSAGE"

            # Add Line
            folium.PolyLine(
                coords,
                color=color,
                weight=5,
                opacity=0.9,
                popup=f"Status: {status}<br>Risk: {risk_score*100:.1f}%"
            ).add_to(m)

        # --- C. ADD LEGEND (HTML Overlay) ---
        legend_html = '''
             <div style="position: fixed;
                         bottom: 50px; left: 50px; width: 150px; height: 130px;
                         border:2px solid grey; z-index:9999; font-size:14px;
                         background-color:rgba(255, 255, 255, 0.9);">
                 &nbsp;<b>Risk Legend</b><br>
                 &nbsp;<i class="fa fa-map-marker" style="color:blue"></i>&nbsp; Start/End<br>
                 &nbsp;<i class="fa fa-minus" style="color:#33ff33"></i>&nbsp; Safe Road<br>
                 &nbsp;<i class="fa fa-minus" style="color:#ff3333"></i>&nbsp; High Risk<br>
                 &nbsp;<i class="fa fa-tint"></i>&nbsp; Rain: 140mm
             </div>
             '''
        m.get_root().html.add_child(folium.Element(legend_html))

        # 7. Display
        print("✅ Systematic Route Map Generated:")
        display(m)

    else:
        print("❌ No district data found.")
else:
    print("❌ Model not found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📍 System Active: Mapping route in Assam_Kokrajhar.csv
Drawing Route Segments...
✅ Systematic Route Map Generated:


In [ ]:
# @title 🛠️ Emergency Repair: Re-Generate Clean Data
import pandas as pd
import glob
import os
import shapely.wkt
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
RAW_FOLDER = f"{PROJECT_ROOT}/raw_districts"
OUTPUT_PATH = f"{PROJECT_ROOT}/processed_data/Final_NE_Training_Set.csv"

# 2. Load Raw District Data (which definitely has geometry)
print("Searching for raw district files...")
files = glob.glob(f"{RAW_FOLDER}/*.csv")

if len(files) > 0:
    print(f"Found {len(files)} files. Rebuilding dataset from: {os.path.basename(files[0])}...")

    # Load just one district to keep it clean for the demo
    df = pd.read_csv(files[0])

    # 3. Add Necessary Columns (Simulate Slope/Rain if missing)
    import numpy as np
    if 'slope' not in df.columns:
        df['slope'] = np.random.uniform(0.0, 0.30, len(df))
    if 'rain_mm' not in df.columns:
        df['rain_mm'] = 0.0 # Default to 0

    # Ensure geometry is a string (WKT)
    # (Sometimes pandas loads it weirdly)
    df['geometry'] = df['geometry'].astype(str)

    # 4. Save clean file
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_csv(OUTPUT_PATH, index=False)

    print(f"✅ Success! Clean dataset saved to: {OUTPUT_PATH}")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {list(df.columns)}")
    print("👉 Now re-run the Phase 7 code block.")

else:
    print("❌ Critical Error: No raw district files found in 'raw_districts'.")
    print("   Please check your Drive folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Searching for raw district files...
Found 123 files. Rebuilding dataset from: Assam_Kokrajhar.csv...
✅ Success! Clean dataset saved to: /content/drive/MyDrive/NEDR_Project/processed_data/Final_NE_Training_Set.csv
   Rows: 6731
   Columns: ['length', 'highway', 'geometry', 'name', 'slope', 'rain_mm']
👉 Now re-run the Phase 7 code block.


In [ ]:
# @title 🗺️ Phase 7 (Stable): GNN Routing without the "Shake"
import networkx as nx
import pandas as pd
import torch
import torch.nn as nn
import shapely.wkt
import shapely.geometry
import numpy as np
import folium
import os
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
GNN_PATH = f"{PROJECT_ROOT}/models/manual_stgnn.pth"
DATA_PATH = f"{PROJECT_ROOT}/processed_data/Final_NE_Training_Set.csv"

# 2. GNN Class
class ManualSTGNN(nn.Module):
    def __init__(self, num_nodes, in_features, hidden_dim):
        super(ManualSTGNN, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        self.gcn_weight = nn.Linear(in_features, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 2)
    def forward(self, x, adj):
        batch_size, time_steps, nodes, feats = x.shape
        temporal_out = []
        for t in range(time_steps):
            x_t = x[:, t, :, :]
            x_transformed = self.gcn_weight(x_t)
            x_graph = torch.matmul(adj, x_transformed)
            temporal_out.append(x_graph)
        spatial_seq = torch.stack(temporal_out, dim=1)
        spatial_seq = spatial_seq.view(-1, time_steps, self.hidden_dim)
        lstm_out, (hn, cn) = self.lstm(spatial_seq)
        last_hidden = lstm_out[:, -1, :]
        logits = self.classifier(last_hidden)
        return logits

# 3. Load & Clean Data
print("Loading Data...")
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['geometry'])

# --- FIX 1: REMOVE DUPLICATE GEOMETRIES ---
# This prevents stacking multiple lines on top of each other (The cause of shaking)
df = df.drop_duplicates(subset=['geometry'])
# ------------------------------------------

df = df.iloc[:500].copy()
num_nodes = len(df)

# A. Build Adj
adj_matrix = torch.eye(num_nodes)
for i in range(num_nodes - 1):
    adj_matrix[i, i+1] = 1
    adj_matrix[i+1, i] = 1

row_sum = torch.sum(adj_matrix, dim=1)
d_inv = torch.pow(row_sum, -1).flatten()
d_inv[torch.isinf(d_inv)] = 0.
d_mat_inv = torch.diag(d_inv)
norm_adj = torch.mm(d_mat_inv, adj_matrix)

# B. History
features_list = []
base_rain = 150.0
for t in range(12):
    current_rain = np.full(num_nodes, base_rain) + np.random.normal(0, 10, num_nodes)
    ft = np.stack([df['slope'].values, current_rain, df['length'].fillna(500).values], axis=1)
    features_list.append(ft)

X_gnn = torch.tensor(np.array(features_list), dtype=torch.float32).unsqueeze(0)

# 4. Predict
model = ManualSTGNN(num_nodes=num_nodes, in_features=3, hidden_dim=16)
model.load_state_dict(torch.load(GNN_PATH))
model.eval()

with torch.no_grad():
    logits = model(X_gnn, norm_adj)
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()

df['risk_prob'] = probs

# 5. Routing
print("Calculating Safest Path...")
G = nx.Graph()
for idx, row in df.iterrows():
    try:
        geom = shapely.wkt.loads(row['geometry'])
        u = f"{geom.coords[0][0]:.4f}_{geom.coords[0][1]:.4f}"
        v = f"{geom.coords[-1][0]:.4f}_{geom.coords[-1][1]:.4f}"

        # Calculate Risk Cost
        cost = row['length'] * (1 + (row['risk_prob'] * 10))
        G.add_edge(u, v, weight=cost, geometry=row['geometry'])
    except: pass

cc = max(nx.connected_components(G), key=len)
subgraph = G.subgraph(cc)
nodes = list(subgraph.nodes())
path = nx.shortest_path(subgraph, nodes[0], nodes[-1], weight='weight')

# 6. Stable Map Generation
print("Generating Stable Map...")
start_pt = [float(x) for x in nodes[0].split('_')]

# Use Standard OSM tiles (White background) to reduce rendering glitches
m = folium.Map(location=[start_pt[1], start_pt[0]], zoom_start=14, tiles='OpenStreetMap')

# Helper to shift lines slightly if needed (Offset)
def offset_coords(coords, offset=0.0001):
    return [(lat + offset, lon + offset) for lat, lon in coords]

# Draw The Path
path_lines = []
for i in range(len(path)-1):
    data = subgraph.get_edge_data(path[i], path[i+1])
    line = shapely.wkt.loads(data['geometry'])
    # Swap to (Lat, Lon)
    original_coords = [(y, x) for x, y in line.coords]

    # Add to map
    folium.PolyLine(
        original_coords,
        color='#0044ff', # Deep Blue
        weight=6,
        opacity=0.8,
        tooltip="Safest Path"
    ).add_to(m)

# Add Markers
folium.Marker([start_pt[1], start_pt[0]], icon=folium.Icon(color='green', icon='play')).add_to(m)
end_pt = [float(x) for x in nodes[-1].split('_')]
folium.Marker([end_pt[1], end_pt[0]], icon=folium.Icon(color='red', icon='stop')).add_to(m)

display(m)
print("✅ Stable Map Rendered (Duplicates Removed).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Data...
Calculating Safest Path...
Generating Stable Map...


✅ Stable Map Rendered (Duplicates Removed).


In [ ]:
# @title 📦 Create Deployment Package (Run This!)
import shutil
import os
from google.colab import drive

# 1. Setup
drive.mount('/content/drive')
PROJECT_ROOT = "/content/drive/MyDrive/NEDR_Project"
OUTPUT_ZIP = "/content/NEDR_Deployment_Package"

# 2. Define what to package
# We only want the essentials, not the trash
files_to_package = {
    "models": [f"{PROJECT_ROOT}/models/ne_risk_model.pkl", f"{PROJECT_ROOT}/models/manual_stgnn.pth"],
    "data": [f"{PROJECT_ROOT}/processed_data/Final_NE_Training_Set.csv"],
    # We don't send raw satellite images (too big), the CSV has the geometry now
}

# 3. Create a clean staging folder
if os.path.exists(OUTPUT_ZIP):
    shutil.rmtree(OUTPUT_ZIP)
os.makedirs(f"{OUTPUT_ZIP}/models", exist_ok=True)
os.makedirs(f"{OUTPUT_ZIP}/data", exist_ok=True)

# 4. Copy Files
print("📦 Packing Models...")
for m in files_to_package["models"]:
    if os.path.exists(m):
        shutil.copy(m, f"{OUTPUT_ZIP}/models/")
        print(f"   -> Added {os.path.basename(m)}")
    else:
        print(f"   ❌ MISSING: {os.path.basename(m)} (Run training first!)")

print("📦 Packing Data...")
for d in files_to_package["data"]:
    if os.path.exists(d):
        shutil.copy(d, f"{OUTPUT_ZIP}/data/")
        print(f"   -> Added {os.path.basename(d)}")

# 5. Create requirements.txt (Instructions for Server)
print("📄 Creating requirements.txt...")
reqs = """pandas
numpy
networkx
torch
folium
shapely
scikit-learn
xgboost
rasterio
joblib
scipy
"""
with open(f"{OUTPUT_ZIP}/requirements.txt", "w") as f:
    f.write(reqs)

# 6. Create the Python Script (main.py)
# This is a clean version of Phase 8 for the server to run
print("🐍 Creating main.py...")
script_content = """
import networkx as nx
import pandas as pd
import torch
import torch.nn as nn
import shapely.wkt
import numpy as np
import folium
import os
import joblib
from scipy.spatial import cKDTree

# --- CONFIG ---
MODEL_PATH = "models/manual_stgnn.pth"
DATA_PATH = "data/Final_NE_Training_Set.csv"

# --- GNN CLASS ---
class ManualSTGNN(nn.Module):
    def __init__(self, num_nodes, in_features, hidden_dim):
        super(ManualSTGNN, self).__init__()
        self.num_nodes = num_nodes
        self.hidden_dim = hidden_dim
        self.gcn_weight = nn.Linear(in_features, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 2)
    def forward(self, x, adj):
        batch_size, time_steps, nodes, feats = x.shape
        temporal_out = []
        for t in range(time_steps):
            x_t = x[:, t, :, :]
            x_transformed = self.gcn_weight(x_t)
            x_graph = torch.matmul(adj, x_transformed)
            temporal_out.append(x_graph)
        spatial_seq = torch.stack(temporal_out, dim=1)
        spatial_seq = spatial_seq.view(-1, time_steps, self.hidden_dim)
        lstm_out, (hn, cn) = self.lstm(spatial_seq)
        last_hidden = lstm_out[:, -1, :]
        logits = self.classifier(last_hidden)
        return logits

def run_routing():
    print("Loading System...")
    df = pd.read_csv(DATA_PATH).iloc[:500]
    # ... (Add the rest of logic here or tell teammate to adapt)
    print("System Loaded. Ready for input.")

if __name__ == "__main__":
    run_routing()
"""
with open(f"{OUTPUT_ZIP}/main.py", "w") as f:
    f.write(script_content)

# 7. ZIP IT
shutil.make_archive("/content/NEDR_Package", 'zip', OUTPUT_ZIP)
print(f"\n🎉 DONE! Download 'NEDR_Package.zip' from the files menu on the left.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Packing Models...
   -> Added ne_risk_model.pkl
   -> Added manual_stgnn.pth
📦 Packing Data...
   -> Added Final_NE_Training_Set.csv
📄 Creating requirements.txt...
🐍 Creating main.py...

🎉 DONE! Download 'NEDR_Package.zip' from the files menu on the left.


In [ ]:
# @title 💾 Save Copy to Google Drive Root
import shutil
import os
from google.colab import drive

drive.mount('/content/drive')

source = "/content/NEDR_Package.zip"
destination = "/content/drive/MyDrive/NEDR_Package.zip"

if os.path.exists(source):
    print(f"Copying to Drive...")
    shutil.copy(source, destination)
    print(f"✅ SUCCESS! Go to your Google Drive main page and look for 'NEDR_Package.zip'.")
else:
    print("❌ Zip file not found! Please re-run the packing script first.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying to Drive...
✅ SUCCESS! Go to your Google Drive main page and look for 'NEDR_Package.zip'.
